# Investigating the rephrasing cababilities of some LLMs

ChatGPT works flawlessly with zero-shot but is expensive so we do not use it here.

There is also:
-  [bigscience/bloom-3b](https://huggingface.co/bigscience/bloom-3b). The large Bloom works great.
- [eugenesiow/bart-paraphrase](https://huggingface.co/eugenesiow/bart-paraphrase)
- [EleutherAI/gpt-j-6B](https://huggingface.co/EleutherAI/gpt-j-6B)

## Setup

In [ ]:
!pip install transformers datasets accelerate sentencepiece -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, T5Tokenizer, T5ForConditionalGeneration

In [ ]:
input_events = """These are the raw events:
From 00:00 to 00:03 / for 3 seconds - A person walking.
From 00:03 to 00:06 / for 3 seconds - Someone waving with their right hand.
From 00:06 to 00:10 / for 4 seconds - A person jumps three times.

A summary of the above events in one sentence each is:
"""

input_events_few_shot = """<chain>
From 00:00 to 00:03 / for 3 seconds - A person walking.
From 00:03 to 00:05 / for 2 seconds - Someone waving with their right hand.
From 00:05 to 00:10 / for 4 seconds - A person jumps three times.
<rephrase>
In the beginning, someone is walking a few steps. Then, from second 3 onwards, a person is waving for 2 seconds. They are doing it with their right hand. Finally, we can see a person jumping three times and stopping at second 10.

<chain>
From 00:00 to 00:04 / for 4 seconds - Someone walking their dog.
From 00:04 to 00:05 / for 1 second  - The person is standing still.
From 00:05 to 00:06 / for 1 second  - Someone waving with their left hand.
From 00:06 to 00:11 / for 5 seconds - A person is making swimming moves with their arms.
<rephrase>

"""

## OPT (by Meta AI)

In [ ]:
# up to 11B should be possible
model_name = "facebook/opt-2.7b"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False) # the fast tokenizer currently does not work correctly

In [ ]:
input_ids = tokenizer(input_events_few_shot, return_tensors="pt").input_ids.cuda()
generated_ids = model.generate(input_ids, do_sample=True, temperature=0.2, max_new_tokens=200, num_return_sequences=5)
outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
for output in outputs:
  print(output)
  print() # newline
  print("----------------------------")
  print() # newline


<chain>
From 00:00 to 00:03 / for 3 seconds - A person walking.
From 00:03 to 00:05 / for 2 seconds - Someone waving with their right hand.
From 00:05 to 00:10 / for 4 seconds - A person jumps three times.
<rephrase>
In the beginning, someone is walking a few steps. Then, from second 3 onwards, a person is waving for 2 seconds. They are doing it with their right hand. Finally, we can see a person jumping three times and stopping at second 10.

<chain>
From 00:00 to 00:04 / for 4 seconds - Someone walking their dog.
From 00:04 to 00:05 / for 1 second  - The person is standing still.
From 00:05 to 00:06 / for 1 second  - Someone waving with their left hand.
From 00:06 to 00:11 / for 5 seconds - A person is making swimming moves with their arms.
<rephrase>
In the beginning, someone is walking a few steps. Then, from second 3 onwards, a person is making swimming moves with their arms. Finally, we can see a person jumping three times and stopping at second 10.

<chain>
From 00:00 to 00:04 

## Flan-T5 (by Google AI)

In [ ]:
# https://huggingface.co/models?search=flan-t5
model_name = "google/flan-t5-large"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

In [ ]:
input_ids = tokenizer(input_events_few_shot, return_tensors="pt").input_ids.to("cuda")
generated_ids = model.generate(input_ids, do_sample=True, temperature=0.75, max_new_tokens=200, num_return_sequences=5)
outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
for output in outputs:
  print(output)
  print() # newline
  print("----------------------------")
  print() # newline

At first, someone is walking their dog. Then, someone is standing still. Then, someone is waving with their left hand. Finally, a person is making swimming moves with their arms.

----------------------------

From 00:00 to 00:04 the person is walking their dog, then he walks the dog for 4 seconds. From 00:04 to 00:05 the person is standing still, then he walks the dog for 1 second. From 00:05 to 00:06 the person is standing still, then he walks the dog for 1 second. From 00:06 to 00:11 the person is swimming with their arms.

----------------------------

In the beginning, someone walking their dog. They are standing still for 1 second. From 00:05 to 00:06 / for 1 second - Someone waving with their left hand. From 00:06 to 00:11 / for 5 seconds - A person is making swimming moves with their arms.

----------------------------

In the beginning, someone is walking their dog. From 00:04 to 00:05 / for 1 second - The person is standing still. From 00:05 to 00:06 / for 1 second - Someone 

## GPT-J
https://huggingface.co/EleutherAI/gpt-j-6B

In [ ]:
model_name = "EleutherAI/gpt-j-6B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)

In [ ]:
input_ids = tokenizer(input_events_few_shot, return_tensors="pt").input_ids.to("cuda")
generated_ids = model.generate(input_ids, do_sample=True, temperature=0.75, max_new_tokens=200, num_return_sequences=5)
outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
for output in outputs:
  print(output)
  print() # newline
  print("----------------------------")
  print() # newline